In [9]:
import json
from datetime import datetime, timezone

import pandas as pd
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()

TABLE_NAME = "imerg"
CONTAINER = "rasterstats-public"
PARTITION_COL = "iso3"
STAGE = "dev"

In [10]:
def get_iso3_list(table_name: str) -> list[str]:
    """Fetch the distinct iso3 values from the table."""
    engine = stratus.get_engine("prod")
    df = pd.read_sql(f"SELECT DISTINCT {PARTITION_COL} FROM {table_name} ORDER BY {PARTITION_COL}", engine)
    return df[PARTITION_COL].tolist()


def get_table_stats(table_name: str) -> dict:
    """Compute metadata stats directly from the database."""
    engine = stratus.get_engine("prod")
    row = pd.read_sql(f"""
        SELECT
            COUNT(*)                        AS row_count,
            COUNT(DISTINCT {PARTITION_COL}) AS partition_count,
            MIN(valid_date)                 AS valid_date_min,
            MAX(valid_date)                 AS valid_date_max
        FROM {table_name}
    """, engine).iloc[0]

    countries = pd.read_sql(
        f"SELECT DISTINCT {PARTITION_COL} FROM {table_name} ORDER BY {PARTITION_COL}", engine
    )[PARTITION_COL].tolist()

    columns = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 0", engine).columns.tolist()

    return {
        "row_count": int(row["row_count"]),
        "countries": countries,
        "valid_date_min": row["valid_date_min"],
        "valid_date_max": row["valid_date_max"],
        "columns": columns,
    }


def read_partition(table_name: str, iso3: str) -> pd.DataFrame:
    """Read a single iso3 partition from PostgreSQL."""
    engine = stratus.get_engine("prod")
    df = pd.read_sql(f"SELECT * FROM {table_name} WHERE {PARTITION_COL} = %(iso3)s", engine, params={"iso3": iso3})
    return df


def write_partition(
    df_partition: pd.DataFrame,
    iso3: str,
    table_name: str,
) -> int:
    """Write a single iso3 partition to Blob Storage as Parquet."""
    blob_path = f"{table_name}/{PARTITION_COL}={iso3}/data.parquet"
    stratus.upload_parquet_to_blob(df_partition, blob_path, stage=STAGE, container_name=CONTAINER)
    return len(df_partition)


def write_metadata(
    container_client,
    table_name: str,
    stats: dict,
) -> None:
    """Write a _metadata.json file alongside the partitions."""
    meta = {
        "table": table_name,
        "last_updated": datetime.now(timezone.utc).isoformat(),
        "row_count": stats["row_count"],
        "partition_count": len(stats["countries"]),
        "temporal_extent": {
            "valid_date_min": str(stats["valid_date_min"]),
            "valid_date_max": str(stats["valid_date_max"]),
        },
        "columns": stats["columns"],
        "countries": stats["countries"],
    }
    blob_client = container_client.get_blob_client(f"{table_name}/_metadata.json")
    blob_client.upload_blob(
        json.dumps(meta, indent=2).encode(),
        overwrite=True,
    )
    print(f"  Metadata written to {table_name}/_metadata.json")

In [11]:
stats = get_table_stats(TABLE_NAME)
iso3_list = stats["countries"]
container_client = stratus.get_container_client(CONTAINER, STAGE, write=True)

print(f"Writing {len(iso3_list)} partitions to {CONTAINER}/{TABLE_NAME}/")
total_rows = 0
for iso3 in iso3_list:
    df = read_partition(TABLE_NAME, iso3)
    rows = write_partition(df, iso3, TABLE_NAME)
    total_rows += rows
    print(f"  {iso3}: {rows:,} rows")

Writing 154 partitions to rasterstats-public/imerg/
  AFG: 4,503,444 rows
  AGO: 196,251 rows
  ALB: 134,277 rows
  ARE: 82,632 rows
  ARG: 258,225 rows
  ARM: 123,948 rows
  ATG: 92,961 rows
  AZE: 774,675 rows
  BDI: 196,308 rows
  BEN: 134,277 rows
  BES: 41,316 rows
  BFA: 609,411 rows
  BGD: 92,961 rows
  BGR: 299,541 rows
  BHS: 340,857 rows
  BLR: 82,632 rows
  BLZ: 72,303 rows
  BMU: 103,290 rows
  BOL: 103,290 rows
  BRA: 289,212 rows
  BRB: 123,948 rows
  BTN: 216,909 rows
  BWA: 185,922 rows
  CAF: 929,610 rows
  CHL: 175,576 rows
  CHN: 361,480 rows
  CIV: 351,152 rows
  CMR: 712,632 rows
  COD: 2,230,848 rows
  COG: 134,264 rows
  COL: 11,939,168 rows
  COM: 41,312 rows
  CPV: 237,544 rows
  CRI: 82,624 rows
  CUB: 175,576 rows
  CYM: 72,296 rows
  DJI: 72,296 rows
  DMA: 113,608 rows
  DOM: 113,608 rows
  DZA: 506,072 rows
  ECU: 268,528 rows
  EGY: 289,184 rows
  ERI: 72,296 rows
  ESH: 51,640 rows
  ETH: 1,094,768 rows
  FJI: 51,640 rows
  FSM: 51,640 rows
  GAB: 103,28

In [14]:
write_metadata(container_client, TABLE_NAME, stats)
print(f"Done — {stats['row_count']:,} rows written across {len(stats['countries'])} partitions")

  Metadata written to imerg/_metadata.json
Done — 80,925,208 rows written across 154 partitions
